# Scenario: Auditing Age Bias in a Stroke Prediction AI

In [16]:
import pandas as pd
import sqlite3
# creating dataset representing the AI's validation testing pool
eval_data = {
    "patient_id": ["P-001", "P-002", "P-003", "P-004", "P-005", "P-006", "P-007", "P-008"],
    "age": [24, 28, 35, 78, 31, 82, 42, 19],
    "stroke_occurred": [0, 0, 0, 1, 0, 1, 0, 0], # 1 = Yes, 0 = No
    "ai_prediction": [0, 0, 0, 0, 0, 1, 0, 0] # Look closely at P-004 (Seniors)!
}
# adding the dataset into DataFrame
df_eval_data = pd.DataFrame(eval_data)
# creating sql and connecting to save temp memory
connt = sqlite3.connect(":memory:")
df_eval_data.to_sql("ai_validation_pool", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("********************************* Demographic Bias Audit Database is ready! **************")

********************************* Demographic Bias Audit Database is ready! **************


# The Demographic Breakdown

In [17]:
# query for all data to review
all_data = "SELECT * FROM ai_validation_pool"
print("*********************************** all data to reveiw ****************")
display(run_query(all_data))
"""SQL query to categorize the patients into two age buckets: 'Under 65' and '65 and Over'. 
Count how many total patients fall into each bucket, and how many actual stroke cases (stroke_occurred = 1) occurred in each."""
stroke_frequency = """
SELECT
    CASE
        WHEN age < 65 THEN 'Under 65'
        ELSE '65 and Over'
        END AS age_bucket,
        COUNT(*) AS total_patients,
        SUM(CASE WHEN stroke_occurred = 1 THEN 1 ELSE 0 END) AS stroke_cases
FROM ai_validation_pool
GROUP BY age_bucket        
"""
print("********************************** strok frequency among diffrent ages ***************")
display(run_query(stroke_frequency))

*********************************** all data to reveiw ****************


,patient_id,age,stroke_occurred,ai_prediction
0,P-001,24,0,0
1,P-002,28,0,0
2,P-003,35,0,0
3,P-004,78,1,0
4,P-005,31,0,0
5,P-006,82,1,1
6,P-007,42,0,0
7,P-008,19,0,0


********************************** strok frequency among diffrent ages ***************


,age_bucket,total_patients,stroke_cases
0,65 and Over,2,2
1,Under 65,6,0


# Calculating Age-Specific False Negatives

In [18]:
""" SQL query to find rows where the AI made a dangerous mistake: 
it predicted no stroke (ai_prediction = 0) but a stroke actually occurred (stroke_occurred = 1). 
Include the patient's age """
age_specific_negatives = """
SELECT
    patient_id, 
    age, 
    stroke_occurred,
    ai_prediction
FROM ai_validation_pool
WHERE stroke_occurred = 1 AND ai_prediction = 0;
"""
print("************************************* Calculatting Age_Specific False Negatives **********")
display(run_query(age_specific_negatives))

************************************* Calculatting Age_Specific False Negatives **********


,patient_id,age,stroke_occurred,ai_prediction
0,P-004,78,1,0
